# Lab Final : เว็บทดสอบโมเดล (Polygon)
notebook นี้อธิบายว่าเว็บทำงานยังไง ตั้งแต่หลังบ้านจนขึ้นอินเทอร์เน็ต โดยเว็บใช้ **โมเดล Polygon (segment) อย่างเดียว**

🌐 **เว็บออนไลน์:** https://witthaya22-final-dip.static.hf.space

### ภาพรวม : เว็บทำนายได้ 2 วิธี
```
 วิธีที่ 1 : รันในเครื่อง (localhost)                    วิธีที่ 2 : ออนไลน์ (Hugging Face, ฟรี)
 ┌────────────┐  POST /api/predict  ┌──────────────┐     ┌──────────────────────────────────┐
 │ หน้าบ้าน     │ ──── ส่งภาพ ─────▶ │ หลังบ้าน       │     │ หน้าบ้าน + ไฟล์โมเดล poly.onnx        │
 │ (เบราว์เซอร์) │ ◀──── JSON ──────── │ FastAPI+YOLO │     │ เบราว์เซอร์รันโมเดลเอง (infer.js)      │
 └────────────┘                     └──────────────┘     └──────────────────────────────────┘
       ใช้ weights/poly_best.pt                                ใช้ web/frontend/models/poly.onnx
```
หน้าเว็บ **เลือกเองอัตโนมัติ** ถ้าเจอหลังบ้านจะใช้วิธีที่ 1 ถ้าไม่เจอ (เช่น บน Hugging Face) จะใช้วิธีที่ 2

### ไฟล์ของเว็บ
| ไฟล์ | หน้าที่ |
|---|---|
| `web/backend/app.py` | **หลังบ้าน** REST API ด้วย FastAPI โหลดโมเดลแล้วรับภาพมาทำนาย |
| `web/frontend/index.html` | โครงหน้าเว็บ (กล่องลากภาพ, slider ปรับ confidence, ปุ่ม Predict, พื้นที่แสดงผล) |
| `web/frontend/app.js` | ควบคุมหน้าเว็บ: เช็คหลังบ้าน, รับภาพ, เรียกโมเดล, วาดผล |
| `web/frontend/infer.js` | รันโมเดล ONNX **ในเบราว์เซอร์** (ใช้ตอนไม่มีหลังบ้าน) |
| `web/frontend/config.js` | ตั้งค่าว่าจะใช้หลังบ้าน หรือรันในเบราว์เซอร์ |
| `web/frontend/style.css` | หน้าตาเว็บ (รองรับมือถือและ dark mode) |
| `web/frontend/models/poly.onnx` | โมเดล Polygon ที่แปลงเป็น ONNX แล้ว (สร้างใน `lab_final_poly.ipynb` ขั้น 6) |

> ⚠️ เปิด notebook นี้จากโฟลเดอร์ `final_dip/` และต้องเทรน/export โมเดลใน `lab_final_poly.ipynb` ก่อน

**อธิบาย Cell 1 :** import ไลบรารีที่ใช้ในการอธิบาย
- `subprocess` ใช้เปิด server หลังบ้านเป็นโปรแกรมแยก
- `requests` ใช้เรียก API แบบเดียวกับที่หน้าเว็บทำ
- `onnxruntime` ใช้รันไฟล์ `.onnx` (เพื่อจำลองสิ่งที่เบราว์เซอร์ทำ)
- `cv2`, `numpy`, `matplotlib` ใช้จัดการและแสดงภาพ

In [ ]:
# ===== Cell 1 : import =====
import os, sys, json, time, subprocess
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"     # แก้ error OMP บน Windows
import requests
import numpy as np, cv2
import matplotlib.pyplot as plt
import onnxruntime as ort

TEST_IMG = "datasets/yolo_poly/test/images/car_013.jpg"   # ภาพที่ใช้ทดสอบตลอด notebook
COLORS = {"airplane": (47, 125, 225), "boat": (17, 165, 154), "car": (232, 88, 45), "motorcycle": (166, 79, 214)}
print("ภาพทดสอบ:", TEST_IMG)

---
# Part 1 : หลังบ้าน (Backend) – `web/backend/app.py`

**FastAPI** คือไลบรารี Python สำหรับทำ REST API (ช่องทางให้โปรแกรมอื่นส่งข้อมูลมาขอผลลัพธ์ผ่าน HTTP)

หลังบ้านทำงานแบบนี้
1. ตอนเปิด server จะ **โหลดโมเดล `weights/poly_best.pt` ครั้งเดียว** เก็บไว้ในหน่วยความจำ ถ้าโหลดใหม่ทุกครั้งที่มีคนส่งภาพจะช้ามาก
2. เปิดช่องทาง (endpoint) ไว้ 3 ทาง
   | method + path | ใช้ทำอะไร |
   |---|---|
   | `GET /api/health` | เช็คว่า server พร้อมไหม โหลดโมเดลได้หรือยัง |
   | `POST /api/predict` | ส่งภาพมาให้ทำนาย ได้ผลกลับเป็น JSON |
   | `GET /` | ส่งหน้าเว็บ (index.html ฯลฯ) ให้เบราว์เซอร์ หน้าบ้านกับหลังบ้านจึงอยู่ที่เดียวกัน |
3. เปิด **CORS** ไว้ หน้าเว็บที่อยู่คนละโดเมนจึงเรียก API นี้ได้

**อธิบาย Cell 2 :** แสดงโค้ดส่วน `/api/predict` จากไฟล์จริง จะเห็นว่ามี 3 ขั้น
- **(1) ตรวจสอบ input:** มีโมเดลไหม, ไฟล์ใหญ่เกิน 10 MB ไหม, เปิดเป็นภาพได้ไหม (`exif_transpose` หมุนภาพจากมือถือให้ตั้งตรง)
- **(2) ส่งภาพให้ YOLO:** แปลงสี RGB เป็น BGR แบบ OpenCV แล้วเรียก `model.predict(...)` และจับเวลาไปด้วย
- **(3) แปลงผลเป็น JSON:** ดึงชื่อคลาส ความมั่นใจ กรอบ และจุด polygon (`result.masks.xy`) ของแต่ละวัตถุ แล้วเรียงจากมั่นใจมากไปน้อย

In [ ]:
# ===== Cell 2 : ดูโค้ด /api/predict ในไฟล์ app.py =====
src = open("web/backend/app.py", encoding="utf-8").read()
start = src.index('@app.post("/api/predict")')
end = src.index("# ----", start)
print(src[start:end])

**อธิบาย Cell 3 :** เปิด server หลังบ้านด้วยคำสั่ง `uvicorn web.backend.app:app --port 8000`
- `uvicorn` คือตัวรัน server ของ FastAPI
- `web.backend.app:app` = ไฟล์ `web/backend/app.py` ตัวแปรชื่อ `app`
- เปิดแบบ `subprocess.Popen` ทำให้ server รันแยกไปเลย notebook ใช้ต่อได้
- รอจนกว่า `/api/health` จะตอบกลับ (ใช้เวลาโหลดโมเดลไม่กี่วินาที)

ระหว่างที่ server เปิดอยู่ เข้าดูในเบราว์เซอร์ได้
- http://localhost:8000 → หน้าเว็บ (ใช้หลังบ้านทำนาย)
- http://localhost:8000/docs → **หน้าเอกสาร API อัตโนมัติ** กด "Try it out" แล้วอัปโหลดภาพลองได้เลย

In [ ]:
# ===== Cell 3 : เปิด server หลังบ้าน =====
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "web.backend.app:app", "--port", "8000"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):                     # รอสูงสุด 60 วินาที
    try:
        requests.get("http://localhost:8000/api/health", timeout=1)
        break
    except requests.ConnectionError:
        time.sleep(1)
print("server พร้อมแล้ว -> http://localhost:8000  และ  http://localhost:8000/docs")

**อธิบาย Cell 4 : `GET /api/health`** หน้าเว็บเรียกอันนี้เป็นอย่างแรกตอนเปิด ถ้าได้ `"models": ["poly"]` แปลว่าหลังบ้านพร้อม หน้าเว็บจะใช้หลังบ้านทำนาย

In [ ]:
# ===== Cell 4 : เช็คสถานะ server =====
r = requests.get("http://localhost:8000/api/health")
print("status code:", r.status_code)        # 200 = สำเร็จ
print(json.dumps(r.json(), indent=2))

**อธิบาย Cell 5 : `POST /api/predict`** ส่งภาพแบบ **multipart/form-data** (แบบเดียวกับฟอร์มอัปโหลดไฟล์ในเว็บทั่วไป)
- `files={"file": ...}` = ไฟล์ภาพ
- `data={"conf": 0.25}` = ความมั่นใจขั้นต่ำ ผลที่ต่ำกว่านี้จะถูกตัดทิ้ง

JSON ที่ได้กลับมา
| key | ความหมาย |
|---|---|
| `image_width`, `image_height` | ขนาดภาพ |
| `inference_ms` | เวลาที่โมเดลใช้ทำนาย (มิลลิวินาที) |
| `count` | จำนวนวัตถุที่เจอ |
| `top_class` | คลาสของวัตถุที่มั่นใจที่สุด = คำตอบหลักของภาพ |
| `detections[]` | รายการวัตถุ แต่ละตัวมี `class_name`, `confidence`, `box` [x1,y1,x2,y2], `polygon` [[x,y],...] |

In [ ]:
# ===== Cell 5 : ส่งภาพไปทำนาย =====
with open(TEST_IMG, "rb") as f:
    r = requests.post("http://localhost:8000/api/predict", files={"file": f}, data={"conf": 0.25})
out = r.json()

show = json.loads(json.dumps(out))                     # copy ไว้แสดงผล
for d in show["detections"]:                           # polygon ยาวมาก ตัดให้เหลือ 3 จุดตอนแสดง
    d["polygon"] = d["polygon"][:3] + [f"... อีก {len(d['polygon']) - 3} จุด"]
print(json.dumps(show, indent=2, ensure_ascii=False))

**อธิบาย Cell 6 :** หน้าเว็บเอา JSON ไปวาดผลยังไง (ฟังก์ชัน `renderResult` ใน `app.js`) เขียนเป็น Python ให้ดู
1. วาดภาพต้นฉบับ
2. แต่ละวัตถุ: เอาจุด `polygon` มา **ระบายสีโปร่งแสง** ตามสีประจำคลาส แล้วลากเส้นขอบ
3. วาดป้ายชื่อ เช่น `car 94%` ที่มุมซ้ายบนของกรอบ
4. ใต้ภาพมีตารางรายชื่อวัตถุพร้อมแถบความมั่นใจ

In [ ]:
# ===== Cell 6 : วาดผลจาก JSON (แบบเดียวกับ app.js) =====
img = cv2.cvtColor(cv2.imread(TEST_IMG), cv2.COLOR_BGR2RGB)
overlay = img.copy()
for d in reversed(out["detections"]):                 # วาดตัวที่มั่นใจที่สุดไว้บนสุด
    color = COLORS[d["class_name"]]
    pts = np.array(d["polygon"], np.int32)
    cv2.fillPoly(overlay, [pts], color)                # ระบายสีในรูป polygon
    cv2.polylines(img, [pts], True, color, 3)          # เส้นขอบ
result = cv2.addWeighted(overlay, 0.35, img, 0.65, 0)  # ผสมสีให้โปร่งแสง ~35%
for d in out["detections"]:
    x1, y1 = int(d["box"][0]), int(d["box"][1])
    label = f"{d['class_name']} {d['confidence']*100:.0f}%"
    cv2.rectangle(result, (x1, max(y1 - 26, 0)), (x1 + 12 * len(label), max(y1, 26)), COLORS[d["class_name"]], -1)
    cv2.putText(result, label, (x1 + 4, max(y1 - 7, 19)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

plt.figure(figsize=(11, 7)); plt.imshow(result); plt.axis("off")
plt.title(f"top_class = {out['top_class']}  ({out['count']} objects, {out['inference_ms']} ms)"); plt.show()

---
# Part 2 : หน้าบ้าน (Frontend) – `index.html` + `app.js`

**`index.html`** โครงหน้าเว็บแบ่งเป็น 3 ส่วน
- **ส่วนหัว:** ชื่อเว็บ และป้ายสถานะ (ใช้หลังบ้าน หรือรันในเบราว์เซอร์)
- **แผงซ้าย:** ① กล่องลากภาพหรือคลิกเลือกไฟล์ ② slider ปรับ confidence ③ ปุ่ม Predict
- **แผงขวา:** การ์ดแสดงผล (ภาพที่ระบายสี polygon และตารางวัตถุ)

**`app.js`** ลำดับการทำงาน
| ขั้น | ฟังก์ชัน | ทำอะไร |
|---|---|---|
| 1 | `checkHealth()` | ตอนเปิดหน้า ลองเรียก `/api/health` ถ้าตอบได้ใช้ **โหมด api** ถ้าไม่ได้ใช้ **โหมด browser** |
| 2 | `setImage()` | ผู้ใช้ลากหรือเลือกภาพ แล้วแสดงภาพรอไว้ และเปิดให้กดปุ่ม Predict |
| 3 | `predict()` | โหมด api: ส่งภาพไป `POST /api/predict` · โหมด browser: เรียก `Infer.predict()` ใน `infer.js` |
| 4 | `renderResult()` | วาดผลลงบน `<canvas>` และทำตาราง (เหมือน Cell 6) |

ทั้งสองโหมดคืน **JSON หน้าตาเดียวกัน** จึงใช้โค้ดวาดผลชุดเดียวกัน ต่างกันแค่โหมด browser ส่ง mask เป็นภาพ ส่วนโหมด api ส่งเป็นจุด polygon

**อธิบาย Cell 7 :** `config.js` คือไฟล์ตั้งค่า
- `INFERENCE = "auto"` (ค่าเริ่มต้น) ลองใช้หลังบ้านก่อน ถ้าไม่มีให้รันในเบราว์เซอร์
- `API_BASE = ""` เรียกหลังบ้านที่ server เดียวกับหน้าเว็บ

In [ ]:
# ===== Cell 7 : ดูไฟล์ตั้งค่า config.js =====
print(open("web/frontend/config.js", encoding="utf-8").read())

**อธิบาย Cell 8 :** ปิด server หลังบ้าน (ส่วนต่อไปจะจำลองโหมด browser ซึ่งไม่ใช้หลังบ้าน)

In [ ]:
# ===== Cell 8 : ปิด server =====
server.terminate()
print("ปิด server แล้ว")

---
# Part 3 : รันโมเดลในเบราว์เซอร์ – `infer.js` (เขียนเป็น Python ให้ดูทีละขั้น)

**ทำไมต้องรันในเบราว์เซอร์?** Hugging Face แบบฟรีให้วางแค่ไฟล์เว็บ (static) รัน Python ไม่ได้ เราจึงแปลงโมเดลเป็น **ONNX** แล้วใช้ **ONNX Runtime Web** (JavaScript) รันบนเครื่องของคนที่เปิดเว็บ
- ข้อดี: ฟรี ไม่ต้องมี server เว็บไม่หลับ
- ครั้งแรกที่กด Predict ต้องโหลดไฟล์ `poly.onnx` (~11 MB) ก่อน

ultralytics ใน Python ทำขั้นตอนพวกนี้ให้อัตโนมัติ แต่ในเบราว์เซอร์ไม่มี ultralytics เราจึงเขียน **ทุกขั้นเอง** ใน `infer.js`
1. **letterbox** – ย่อภาพแล้วเติมขอบเทาให้เป็น 512×512
2. **tensor** – แปลงเป็นตัวเลข `[1, 3, 512, 512]` ค่า 0–1
3. **รันโมเดล** – ได้ `output0` (กล่อง + คะแนน + สัมประสิทธิ์ mask) และ `output1` (mask พื้นฐาน)
4. **กรอง conf + NMS** – ตัดกล่องที่ไม่มั่นใจ และกล่องที่ซ้อนกัน
5. **แปลงพิกัดกลับ** – จากภาพ 512×512 เป็นพิกัดของภาพจริง
6. **สร้าง mask** – ผสม mask พื้นฐานตามสัมประสิทธิ์ของแต่ละวัตถุ

Cell ต่อไปนี้ทำตามขั้นตอนเดียวกับ `infer.js` ทุกบรรทัด

**อธิบาย Cell 9 : ขั้น 1 letterbox**
- โมเดลรับภาพขนาด **512×512** เท่านั้น แต่ภาพจริงมีหลายขนาด
- ย่อภาพให้ด้านยาวเท่ากับ 512 โดย **คงสัดส่วนเดิม** (ไม่ยืดภาพ) แล้วเติมขอบสีเทา (114, 114, 114) ให้เต็มสี่เหลี่ยม
- จำค่า `scale` (อัตราย่อ) และ `padX`, `padY` (ความหนาขอบ) ไว้ใช้แปลงพิกัดกลับในขั้น 5

In [ ]:
# ===== Cell 9 : ขั้น 1 letterbox =====
IMGSZ = 512
img = cv2.cvtColor(cv2.imread(TEST_IMG), cv2.COLOR_BGR2RGB)
h0, w0 = img.shape[:2]

scale = min(IMGSZ / w0, IMGSZ / h0)                     # อัตราย่อ (ให้ด้านยาว = 512)
nw, nh = round(w0 * scale), round(h0 * scale)           # ขนาดหลังย่อ
padX = round((IMGSZ - nw) / 2 - 0.1)                    # ขอบซ้าย (สูตรเดียวกับ ultralytics)
padY = round((IMGSZ - nh) / 2 - 0.1)                    # ขอบบน
canvas = np.full((IMGSZ, IMGSZ, 3), 114, np.uint8)      # พื้นสีเทา
canvas[padY:padY + nh, padX:padX + nw] = cv2.resize(img, (nw, nh))

print(f"ภาพจริง {w0}x{h0}  ->  ย่อเป็น {nw}x{nh}  (scale = {scale:.3f})  ขอบ padX = {padX}, padY = {padY}")
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(img); ax[0].set_title(f"original {w0}x{h0}")
ax[1].imshow(canvas); ax[1].set_title("letterbox 512x512")
plt.show()

**อธิบาย Cell 10 : ขั้น 2 + 3 แปลงเป็น tensor แล้วรันโมเดล**
- ภาพเป็นตาราง `[สูง, กว้าง, 3 สี]` แต่โมเดลต้องการ `[1, 3 สี, สูง, กว้าง]` (เรียกว่า CHW) จึงต้องสลับแกนด้วย `transpose`
- หารด้วย 255 ให้ค่าสีอยู่ระหว่าง 0–1
- `sess.run(...)` รันโมเดล ได้ output 2 ตัว
  - `output0 = [1, 40, 5376]` = กล่องที่เป็นไปได้ **5376 กล่อง** แต่ละกล่องมี 40 ค่า: `cx, cy, w, h` (4) + คะแนน 4 คลาส (4) + สัมประสิทธิ์ mask (32)
  - `output1 = [1, 32, 128, 128]` = **mask พื้นฐาน 32 แผ่น** ขนาด 128×128 (เล็กกว่าภาพ 4 เท่า)

In [ ]:
# ===== Cell 10 : ขั้น 2 tensor + ขั้น 3 รันโมเดล =====
x = canvas.transpose(2, 0, 1)[None].astype(np.float32) / 255     # [1, 3, 512, 512] ค่า 0-1
print("input tensor :", x.shape, f"ค่าต่ำสุด {x.min():.2f} สูงสุด {x.max():.2f}")

sess = ort.InferenceSession("web/frontend/models/poly.onnx")
t0 = time.perf_counter()
out0, out1 = sess.run(None, {"images": x})
print(f"รันโมเดลใช้เวลา {(time.perf_counter() - t0) * 1000:.0f} ms")
print("output0 :", out0.shape, " (กล่อง + คะแนน + สัมประสิทธิ์ mask)")
print("output1 :", out1.shape, " (mask พื้นฐาน 32 แผ่น)")

**อธิบาย Cell 11 : ขั้น 4 กรองความมั่นใจ + NMS**
- แต่ละกล่องมีคะแนน 4 คลาส เลือกคลาสที่คะแนนสูงสุด ถ้าต่ำกว่า `conf = 0.25` ตัดทิ้ง
- กล่องที่ผ่านมักซ้อนกันหลายกล่องบนวัตถุเดียว จึงต้องทำ **NMS (Non-Maximum Suppression)**
  - เรียงจากมั่นใจมากไปน้อย
  - ถ้ากล่องคลาสเดียวกันซ้อนกับกล่องที่เก็บไว้แล้วเกิน **IoU 0.7** ให้ตัดทิ้ง
  - **IoU** = พื้นที่ซ้อนกัน ÷ พื้นที่รวมของสองกล่อง (0 = ไม่ซ้อนเลย, 1 = ทับกันพอดี)

In [ ]:
# ===== Cell 11 : ขั้น 4 กรอง conf + NMS =====
names = ["airplane", "boat", "car", "motorcycle"]
pred = out0[0].T                                   # [5376, 40]  1 แถว = 1 กล่อง
boxes, scores, coefs = pred[:, :4], pred[:, 4:8], pred[:, 8:]
cls, conf = scores.argmax(1), scores.max(1)        # คลาสที่คะแนนสูงสุด + คะแนนนั้น

keep = conf >= 0.25
print(f"กล่องทั้งหมด {len(conf)}  ->  ผ่าน conf >= 0.25 : {keep.sum()} กล่อง")
boxes, cls, conf, coefs = boxes[keep], cls[keep], conf[keep], coefs[keep]
xyxy = np.c_[boxes[:, 0] - boxes[:, 2] / 2, boxes[:, 1] - boxes[:, 3] / 2,   # cx,cy,w,h -> x1,y1,x2,y2
             boxes[:, 0] + boxes[:, 2] / 2, boxes[:, 1] + boxes[:, 3] / 2]

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)

final = []
for i in conf.argsort()[::-1]:                     # มั่นใจมาก -> น้อย
    if all(cls[j] != cls[i] or iou(xyxy[j], xyxy[i]) < 0.7 for j in final):
        final.append(i)
print(f"หลัง NMS เหลือ {len(final)} วัตถุ")

**อธิบาย Cell 12 : ขั้น 5 แปลงพิกัดกลับเป็นของภาพจริง**
พิกัดที่ได้เป็นของภาพ 512×512 ที่มีขอบเทา ต้อง **ลบขอบ (padX, padY) ออกก่อน แล้วหารด้วย scale** จึงได้พิกัดบนภาพจริง

In [ ]:
# ===== Cell 12 : ขั้น 5 แปลงพิกัดกลับ =====
for i in final:
    b512 = xyxy[i]
    b = (b512 - [padX, padY, padX, padY]) / scale
    print(f"{names[cls[i]]:<10} conf {conf[i]:.3f} | พิกัดบน 512x512 = {b512.round().astype(int).tolist()}"
          f"  ->  บนภาพจริง = {b.round().astype(int).tolist()}")

**อธิบาย Cell 13 : ขั้น 6 สร้าง mask (รูปร่างวัตถุ)**
- โมเดลไม่ได้ส่ง mask ของแต่ละวัตถุมาตรง ๆ แต่ส่ง **mask พื้นฐาน 32 แผ่น** (output1) กับ **สัมประสิทธิ์ 32 ค่าต่อวัตถุ** (ใน output0)
- mask ของวัตถุ = ผลรวมของ (สัมประสิทธิ์ × mask พื้นฐาน) ทั้ง 32 แผ่น แล้วผ่าน **sigmoid** ถ้าได้มากกว่า 0.5 ถือว่าเป็นส่วนของวัตถุ (เท่ากับเช็คว่าผลรวมมากกว่า 0)
- ตัดให้เหลือเฉพาะในกรอบของวัตถุนั้น (นอกกรอบไม่นับ)
- mask มีขนาด 128×128 จึงตัดขอบเทาออก แล้วขยายกลับเป็นขนาดภาพจริงก่อนระบายสี

ในเบราว์เซอร์ `infer.js` ทำแบบเดียวกัน แล้วให้ `app.js` ระบายสีทับภาพ

In [ ]:
# ===== Cell 13 : ขั้น 6 สร้าง mask แล้วระบายสี =====
protos = out1[0]                                   # [32, 128, 128]
r = protos.shape[1] / IMGSZ                        # 128 / 512 = 0.25
result = img.copy()
fig, axes = plt.subplots(1, len(final) + 1, figsize=(5 * (len(final) + 1), 4.5))
for k, i in enumerate(final):
    m = np.tensordot(coefs[i], protos, axes=1)     # ผสม 32 แผ่นตามสัมประสิทธิ์ -> [128, 128]
    x1, y1, x2, y2 = (xyxy[i] * r).astype(int)
    crop = np.zeros_like(m, bool)
    crop[max(y1, 0):y2, max(x1, 0):x2] = True      # เฉพาะในกรอบ
    mask = (m > 0) & crop                          # sigmoid(m) > 0.5  เท่ากับ  m > 0
    axes[k].imshow(mask, cmap="gray"); axes[k].set_title(f"mask 128x128: {names[cls[i]]}")
    # ตัดขอบเทา แล้วขยายเป็นขนาดภาพจริง
    sub = mask[int(padY * r):int((padY + nh) * r), int(padX * r):int((padX + nw) * r)].astype(np.uint8)
    full = cv2.resize(sub, (w0, h0), interpolation=cv2.INTER_LINEAR).astype(bool)
    result[full] = (0.55 * result[full] + 0.45 * np.array(COLORS[names[cls[i]]])).astype(np.uint8)
axes[-1].imshow(result); axes[-1].set_title("result (what the web shows)")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

**อธิบาย Cell 14 : เทียบผล** ระหว่างโค้ดที่เราเขียนเอง (แบบ `infer.js`) กับ ultralytics ที่รันไฟล์ `.onnx` เดียวกัน ตัวเลขต้องตรงกัน แปลว่าเว็บทำนายได้ถูกต้องเหมือน Python

In [ ]:
# ===== Cell 14 : เทียบกับ ultralytics =====
from ultralytics import YOLO
print("โค้ดของเรา (แบบ infer.js):")
for i in final:
    print(f"   {names[cls[i]]:<10} {conf[i]:.3f}")
res = YOLO("web/frontend/models/poly.onnx", task="segment").predict(TEST_IMG, imgsz=512, conf=0.25, verbose=False)[0]
print("ultralytics (ไฟล์ onnx เดียวกัน):")
for c, s in zip(res.boxes.cls, res.boxes.conf):
    print(f"   {names[int(c)]:<10} {float(s):.3f}")

---
# Part 4 : Deploy ขึ้นอินเทอร์เน็ต (Hugging Face Spaces)

**Hugging Face Spaces** เป็นที่ฝากเว็บฟรี เราใช้แบบ **static** (วางแค่ไฟล์เว็บ) เพราะแบบที่รัน Python/Docker ต้องสมัคร PRO (เสียเงิน)

**สคริปต์ `scripts/05_deploy_huggingface.py` ทำอะไร**
1. สร้าง Space ชื่อ `witthaya22/final-dip` แบบ static (ถ้ามีอยู่แล้วก็ใช้ของเดิม)
2. อัปโหลดทุกไฟล์ใน `web/frontend/` ได้แก่ index.html, app.js, infer.js, config.js, style.css และ `models/poly.onnx`
   - ใช้ `delete_patterns` ลบไฟล์เก่าบน Space ที่ไม่มีในเครื่องแล้ว ให้ Space ตรงกับในเครื่องเสมอ
3. อัปโหลด README ของ Space ที่บอกว่า `sdk: static` และหน้าแรกคือ `index.html`

บน Space ไม่มีหลังบ้าน `/api/health` จึงเรียกไม่ได้ หน้าเว็บเลยสลับไปใช้ **โหมด browser** อัตโนมัติ

**อธิบาย Cell 15 :** ตั้ง `DEPLOY = True` แล้วรัน เพื่ออัปโหลด (ต้อง login ไว้ก่อนด้วย `hf auth login` ครั้งเดียว)

In [ ]:
# ===== Cell 15 : Deploy (ตั้ง DEPLOY = True เพื่ออัปโหลด) =====
DEPLOY = False
if DEPLOY:
    !{sys.executable} scripts/05_deploy_huggingface.py --space witthaya22/final-dip
else:
    print("ยังไม่ deploy (ตั้ง DEPLOY = True แล้วรันใหม่)")

**อธิบาย Cell 16 :** เช็คว่าเว็บออนไลน์และมีไฟล์ครบ (status 200 = เปิดได้)

In [ ]:
# ===== Cell 16 : เช็คเว็บออนไลน์ =====
URL = "https://witthaya22-final-dip.static.hf.space"
for f in ["", "app.js", "infer.js", "config.js", "style.css", "models/poly.onnx"]:
    r = requests.get(f"{URL}/{f}", timeout=60)
    print(f"{r.status_code}  {len(r.content) / 1e3:>9,.0f} KB   /{f}")
print("\nเปิดเว็บ:", URL)